## Homeworkk 1 

### Aggregate daily weather data

We first use groupby to aggregate multiple columns from the hourly data by day:


In [47]:
import pandas as pd

hourly = pd.read_excel("open-meteo-52.55N13.41E38m.xlsx", skiprows=3)
print(hourly)
hourly["timestamp"] = pd.to_datetime(hourly["time"])
#simpler column names
hourly= hourly.rename(columns={
    "temperature_2m (°C)": "temperature",
    "relative_humidity_2m (%)": "humidity",
    "precipitation (mm)": "precipitation",
    "pressure_msl (hPa)": "pressure_sea_level",
    "cloud_cover (%)": "cloud_cover",
    "wind_speed_10m (km/h)": "wind_speed"
})
hourly.head()

                     time  temperature_2m (°C)  weather_code (wmo code)  \
0     2016-10-09 00:00:00                  7.7                        3   
1     2016-10-09 01:00:00                  7.6                        3   
2     2016-10-09 02:00:00                  7.7                        3   
3     2016-10-09 03:00:00                  7.8                       51   
4     2016-10-09 04:00:00                  8.0                       51   
...                   ...                  ...                      ...   
87715 2026-10-11 19:00:00                  9.2                       53   
87716 2026-10-11 20:00:00                  9.6                       51   
87717 2026-10-11 21:00:00                  8.6                       53   
87718 2026-10-11 22:00:00                  8.2                        3   
87719 2026-10-11 23:00:00                  8.1                        1   

       relative_humidity_2m (%)  precipitation (mm)  pressure_msl (hPa)  \
0                       

,time,temperature,weather_code (wmo code),humidity,precipitation,pressure_sea_level,cloud_cover,wind_speed,timestamp
0,2016-10-09 00:00:00,7.7,3,90,0.0,1022.5,92,10.0,2016-10-09 00:00:00
1,2016-10-09 01:00:00,7.6,3,90,0.0,1022.2,91,10.9,2016-10-09 01:00:00
2,2016-10-09 02:00:00,7.7,3,90,0.0,1022.0,100,12.6,2016-10-09 02:00:00
3,2016-10-09 03:00:00,7.8,51,90,0.1,1022.0,100,13.9,2016-10-09 03:00:00
4,2016-10-09 04:00:00,8.0,51,91,0.1,1021.6,100,13.8,2016-10-09 04:00:00


In [48]:
hourly["date"]= hourly["time"].dt.normalize()
hourly["date"]= pd.to_datetime(hourly["date"])
daily = hourly.groupby("date").agg({
    "temperature": "max",
    "humidity": "mean",
    "precipitation": "sum",
    "pressure_sea_level": "mean",
    "cloud_cover": "mean",
    "wind_speed": "mean"
}).reset_index()

daily.columns = [
    "date",
    "max_temperature",
    "humidity",
    "precipitation",
    "pressure",
    "cloud_cover",
    "wind_speed"
]
daily.head()

,date,max_temperature,humidity,precipitation,pressure,cloud_cover,wind_speed
0,2016-10-09,10.7,84.875000,1.1,1022.475000,84.166667,10.366667
1,2016-10-10,11.7,82.000000,0.0,1023.904167,80.458333,5.508333
2,2016-10-11,9.4,91.125000,3.3,1020.887500,99.791667,8.958333
3,2016-10-12,9.0,87.750000,1.7,1022.295833,100.000000,15.425000
4,2016-10-13,7.7,81.916667,0.8,1023.179167,99.791667,15.666667


## Task 1

Use `groupby` to find the average daily precipitation by month

In [49]:
daily["month"] = daily["date"].dt.month
df_monthly_avg_precipitation = daily.groupby("month").agg({"precipitation":"mean"})
print(df_monthly_avg_precipitation)

       precipitation
month               
1           1.822581
2           1.743972
3           1.325484
4           1.058000
5           1.376774
6           2.294333
7           2.459355
8           1.622903
9           1.438000
10          1.698083
11          1.523667
12          1.605806


## Task 2

### Rainy days


Count the number of days with more than 1 mm of precipitation in each month.

In [50]:
daily["rainy_day"] = daily["precipitation"]>1
rainy_day_count = daily.groupby("month").agg({"rainy_day":"sum"})
print(rainy_day_count)

       rainy_day
month           
1            127
2            110
3            100
4             74
5             79
6            104
7            118
8             94
9             76
10           107
11            94
12           111


## Task 3

### Are warm months also dry months?

1. Calculate the average daily maximum temperature for each month.
2. Calculate the average daily precipitation for each month.
3. Merge the two results into one DataFrame.
4. Add month names using a merge.
5. Identify the warmest, wettest, and driest months.

In [53]:
#Calculate average temperature and precipitation by month
df_monthly_avg_temp = daily.groupby("month").agg({"max_temperature":"mean"})
print(df_monthly_avg_temp)
df_merged = pd.merge(df_monthly_avg_precipitation,df_monthly_avg_temp, on=["month"], how = "right")
print(df_merged)

       max_temperature
month                 
1             3.934839
2             6.003901
3            10.052581
4            14.533000
5            19.220000
6            24.832000
7            24.505161
8            25.157742
9            20.628667
10           14.872204
11            8.254333
12            5.163548
       precipitation  max_temperature
month                                
1           1.822581         3.934839
2           1.743972         6.003901
3           1.325484        10.052581
4           1.058000        14.533000
5           1.376774        19.220000
6           2.294333        24.832000
7           2.459355        24.505161
8           1.622903        25.157742
9           1.438000        20.628667
10          1.698083        14.872204
11          1.523667         8.254333
12          1.605806         5.163548


In [60]:
months = pd.DataFrame({
    "month": range(1, 13),
    "month_name": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
})
named_merge = pd.merge(months, df_merged, on= "month", how = "left") 
print(named_merge)

    month month_name  precipitation  max_temperature
0       1        Jan       1.822581         3.934839
1       2        Feb       1.743972         6.003901
2       3        Mar       1.325484        10.052581
3       4        Apr       1.058000        14.533000
4       5        May       1.376774        19.220000
5       6        Jun       2.294333        24.832000
6       7        Jul       2.459355        24.505161
7       8        Aug       1.622903        25.157742
8       9        Sep       1.438000        20.628667
9      10        Oct       1.698083        14.872204
10     11        Nov       1.523667         8.254333
11     12        Dec       1.605806         5.163548


In [64]:
#Find the warmest, wettest, and driest months
warmest = named_merge.loc[named_merge["max_temperature"]==named_merge["max_temperature"].max()]
print(warmest)
wettest =  named_merge.loc[named_merge["precipitation"]==named_merge["precipitation"].max()]
print(wettest)
driest =  named_merge.loc[named_merge["precipitation"]==named_merge["precipitation"].min()]
print(driest)

   month month_name  precipitation  max_temperature
7      8        Aug       1.622903        25.157742
   month month_name  precipitation  max_temperature
6      7        Jul       2.459355        24.505161
   month month_name  precipitation  max_temperature
3      4        Apr          1.058           14.533
